
# 3D flocking — underdamped multi-particle inference

End-to-end demonstration of **underdamped multi-particle** parametric
inference on a 3D flocking-class system with both **position and
velocity** pairwise coupling — a translation-invariant flock held
together by pairwise cohesion plus velocity alignment:

\begin{align}F_p(x, v) = \sum_{q \ne p} \bigl[\,
            k_\mathrm{coh}\,(x_q - x_p)
            + k_\mathrm{alg}\,(v_q - v_p)
          \,\bigr]\end{align}

<div class="alert alert-info"><h4>Note</h4><p>This is an **advanced** example: it uses the parametric estimator
   (:meth:`~SFI.inference.OverdampedLangevinInference.infer_force`) on an interacting multi-particle PSF.  Start with
   the main gallery if you are new to SFI.</p></div>

The two inner solvers of the parametric estimator are run and compared:

1. **Direct Gauss–Newton** (``inner="gn"``) — closed-form normal
   equations on the whitened Gram.  Default for linear-in-$\theta$
   families.
2. **L-BFGS** (``inner="lbfgs"``) — frozen-precision AD minimisation of
   the same whitened objective.  Required for non-linear-in-θ families
   (neural drift, gated interactors); on linear ones it must agree with
   GN to optimiser tolerance — this gallery verifies that parity on a
   full 3D interacting system.

Both solvers use the per-edge
$F.d_x(\mathrm{same\_particle}\!=\!\text{True})$ /
$F.d_v(\mathrm{same\_particle}\!=\!\text{True})$ Jacobian
protocol under the hood (frozen-background approximation), keeping
the per-window cost $\mathcal{O}(N)$ rather than
$\mathcal{O}(N^2)$.


## System: 3D flocking with velocity alignment

$N$ particles in free 3D space, each subject to an isotropic
harmonic trap and two pairwise couplings — **cohesion** (position
spring between particle pairs) and **alignment** (damping between
relative velocities). All three couplings are linear in their
coefficients.



In [ ]:
N_particles = 20
d = 3
dt = 0.005
Nsteps = 3000
oversampling = 4

k_coh_true  = 0.05       # pairwise cohesion (position spring)
k_alg_true  = 0.2        # pairwise velocity alignment (damping)
D_val       = 5e-3       # process noise on velocities

theta_true = np.array([k_coh_true, k_alg_true])

## Simulating the true model

The ground-truth force is purely pairwise (cohesion + velocity
alignment) via :func:`~SFI.statefunc.make_interactor`. ``needs_v=True`` threads
velocities through consistently.

Dropping a confining trap is deliberate: an external harmonic trap
$-k_\mathrm{trap}\, x_p$ is collinear with the cohesion term
whenever the swarm's centre of mass hovers near the origin (both
reduce to $-\text{const}\cdot x_p$), so the two coefficients
are not jointly identifiable. The translation-invariant model below
is well-posed in all parameters.



In [ ]:
from SFI.langevin import UnderdampedProcess
from SFI.statefunc import make_interactor
from SFI.statefunc.params import ParamSpec


def _flock_pair(x, *, v, params):
    dr = x[1] - x[0]
    dv = v[1] - v[0]
    return (params["k_coh"] * dr + params["k_alg"] * dv)[..., None]


F_true_psf = make_interactor(
    _flock_pair, dim=d, rank=1, K=2, n_features=1,
    needs_v=True,
    params=[ParamSpec("k_coh", shape=(), default=float(k_coh_true)),
            ParamSpec("k_alg", shape=(), default=float(k_alg_true))],
).dispatch_pairs(drop_features=True)

# ``ParamSpec.default`` carries the ground-truth values, so the
# simulator binds parameters automatically — no explicit
# ``set_params`` call required.
proc = UnderdampedProcess(F_true_psf, D=D_val)

key = random.PRNGKey(7)
key, kx, kv = random.split(key, 3)
x0 = random.normal(kx, (N_particles, d)) * 1.2
v0 = random.normal(kv, (N_particles, d)) * 0.5
proc.initialize(x0, v0=v0)

key, sub = random.split(key)
t0_sim = time.perf_counter()
coll = proc.simulate(
    dt=dt, Nsteps=Nsteps, key=sub,
    oversampling=oversampling, prerun=200,
)
t_sim = time.perf_counter() - t0_sim

_, X_full, _ = coll.to_arrays(dataset=0)
print(
    f"Simulated {X_full.shape[0]} frames × {X_full.shape[1]} "
    f"particles × {X_full.shape[2]}D in {t_sim:.1f}s"
)

## 3D trajectory visualisation



## Inference setup

Reuse the same 2-parameter interacting PSF as ground truth for both
inference runs. Unknowns: $(k_\mathrm{coh}, k_\mathrm{alg})$.



In [ ]:
F_infer_psf = F_true_psf
D_init = float(D_val)

## Method 1 — direct Gauss–Newton (``inner="gn"``)

The data is clean and the diffusion is known, so we pass both ``D``
and ``Lambda`` — the fixed-noise fast path (no profiling).  With
no measurement noise the skip-trick instrument is unnecessary
(``eiv=False``), and both solvers then minimise the *same* whitened
objective — the parity gate below requires that.



In [ ]:
from SFI import UnderdampedLangevinInference

inf_gn = UnderdampedLangevinInference(coll)
t0 = time.perf_counter()
inf_gn.infer_force(
    F_infer_psf, n_substeps=8,
    Lambda=jnp.zeros((d, d)),
    D=D_init * jnp.eye(d),
    inner="gn", eiv=False, max_outer=8,
)
t_gn = time.perf_counter() - t0

theta_gn = {k: float(np.asarray(v)) for k, v in
            F_infer_psf.unflatten_params(inf_gn.force_coefficients_full).items()}
print(f"[GN] inferred in {t_gn:.1f}s")
print(inf_gn.summary(field="force"))

## Method 2 — L-BFGS (``inner="lbfgs"``)



In [ ]:
inf_loss = UnderdampedLangevinInference(coll)
t0 = time.perf_counter()
inf_loss.infer_force(
    F_infer_psf, n_substeps=8,
    Lambda=jnp.zeros((d, d)),
    D=D_init * jnp.eye(d),
    inner="lbfgs", max_outer=5,
    inner_maxiter=100,
)
t_loss = time.perf_counter() - t0

theta_loss = {k: float(np.asarray(v)) for k, v in
              F_infer_psf.unflatten_params(inf_loss.force_coefficients_full).items()}
print(f"[L-BFGS] inferred in {t_loss:.1f}s")
print(inf_loss.summary(field="force"))

## Recovered couplings — bar chart



In [ ]:
methods = ["GN", "L-BFGS"]
param_names = [r"$k_\mathrm{coh}$", r"$k_\mathrm{alg}$"]

## Relative errors and solver parity



In [ ]:
theta_gn_arr   = np.array([theta_gn["k_coh"],   theta_gn["k_alg"]])
theta_loss_arr = np.array([theta_loss["k_coh"], theta_loss["k_alg"]])

cmp_gn   = inf_gn.compare_params_to_exact(
    {'k_coh': k_coh_true, 'k_alg': k_alg_true}, psf=F_infer_psf
)
cmp_loss = inf_loss.compare_params_to_exact(
    {'k_coh': k_coh_true, 'k_alg': k_alg_true}, psf=F_infer_psf
)
print()
for name, row in cmp_gn.items():
    print(f"  GN   {name}: inferred={row['inferred']:.4g}  rel_error={row['rel_error']:.3e}")
for name, row in cmp_loss.items():
    print(f"  Loss {name}: inferred={row['inferred']:.4g}  rel_error={row['rel_error']:.3e}")
parity = float(np.linalg.norm(theta_loss_arr - theta_gn_arr) / np.linalg.norm(theta_gn_arr))
print(f"  Loss vs GN    : ‖Δθ‖/‖θ‖ = {parity:.3e}  (parity gate)")

## Force-field agreement on held-out samples



## Takeaways

* **Solver parity.** On this linear-in-θ 3D flocking problem the
  **GN** (``inner="gn"``) and **L-BFGS**
  (``inner="lbfgs"``) paths agree to optimiser tolerance
  (``‖Δθ‖/‖θ‖ ≈ 3 × 10⁻³``). Both minimise the same flow-residual
  likelihood; GN exploits its linear-in-θ Gauss–Newton structure,
  the loss path reaches the same minimum via AD + L-BFGS.
* **When to use which.** The L-BFGS solver (``inner="lbfgs"``) is
  required for non-linear-in-θ parametric families (neural drift,
  gated interactors) in multi-particle underdamped systems; GN is
  restricted to linear coefficient recovery but is roughly 4–5×
  faster here.
* **$\mathcal{O}(N)$ approximation.** The multi-particle
  path uses the per-edge
  $F.d_x(\mathrm{same\_particle}\!=\!\text{True})$ /
  $F.d_v(\mathrm{same\_particle}\!=\!\text{True})$ Jacobian
  protocol (frozen-background approximation), keeping the per-
  window cost $\mathcal{O}(N)$ rather than
  $\mathcal{O}(N^2)$. At moderate couplings this introduces
  a mild, systematic downward bias on the recovered coefficients
  — visible here as ``‖Δθ‖/‖θ‖ ≈ 0.28`` vs ground truth — which
  is the price of linear scaling in $N$. Both solvers
  inherit the same bias; their *agreement* is the target of this
  gallery example.



In [ ]:
stamp_output()